# Aadhar pulse : Inferring Life Events and Identity Volatility for Population from Aadhar Data

This notebook operationalizes two coupled ideas:

1. **Aadhaar as a Living Societal Sensor** — identity update *exhaust* is treated as a population-scale pulse signal for life transitions (migration, marriage, schooling pressure, aging stress, shocks).
2. **Identity Entropy Index (IEI)** — a quantitative metric of administrative instability/turbulence derived from update diversity, volume, volatility, and age-structure.

We do **state → district → pincode** analysis and infer **probabilistic life-event hypotheses** at **population level only** (no individual linkage).

## What makes this different
- Uses **every CSV file** in `Datasets/` (biometric + demographic + enrolment; includes the duplicate enrolment folder but de-duplicates rows).
- Builds a **unified long table** (channel × age-band × geography × time) using DuckDB for scale.
- Produces IEI at multiple geographic resolutions and detects **stress spikes** and **change points**.
- Runs a Bayesian-style **Life-Event Inference Engine** + **temporal clustering** of episodes with confidence scoring.

> Recommended: Select the `.venv` kernel (Python 3.9) when running this notebook in VS Code.

> **Attribution / AI mark**: This notebook was assembled with assistance from **GitHub Copilot (GPT-5.2)** and then reviewed/edited by the author.



In [ ]:
# Notebook configuration

from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.preprocessing import RobustScaler

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 150)
np.random.seed(42)

WORKSPACE = Path.cwd()
DATA_DIR = WORKSPACE / 'Datasets'
OUT_DIR = WORKSPACE / 'outputs'
OUT_DIR.mkdir(exist_ok=True)

print('Workspace:', WORKSPACE)
print('Data dir exists:', DATA_DIR.exists())
print('Outputs:', OUT_DIR)


In [ ]:
# 1) Discover *all* CSV files (biometric + demographic + enrolment, incl. enrolment 2)

def glob_sorted(pattern: str) -> list[str]:
    return sorted([str(p) for p in DATA_DIR.glob(pattern)])

FILES = {
    'biometric': glob_sorted('api_data_aadhar_biometric/*.csv'),
    'demographic': glob_sorted('api_data_aadhar_demographic/*.csv'),
    'enrolment': glob_sorted('api_data_aadhar_enrolment/*.csv') + glob_sorted('api_data_aadhar_enrolment 2/*.csv'),
}

for k, v in FILES.items():
    print(f'{k}: {len(v)} files')
    if not v:
        raise FileNotFoundError(f'No files found for {k}')

# Quick file-size sanity
sizes = []
for family, paths in FILES.items():
    for p in paths:
        sizes.append({'family': family, 'file': p, 'mb': Path(p).stat().st_size / 1e6})
sizes_df = pd.DataFrame(sizes).sort_values(['family', 'mb'], ascending=[True, False])
sizes_df.groupby('family').agg(files=('file','count'), total_mb=('mb','sum'), max_mb=('mb','max')).reset_index()

## 2) Ingest at scale with DuckDB
We build a unified long table:

`month, state, district_raw, district_key, pincode, channel, age_band, count, source_file`

Then we aggregate to a **monthly region cube** at state/district/pincode granularity.

In [ ]:
# DuckDB helpers

def _sql_str_list(paths: list[str]) -> str:
    # Convert a Python list of paths into a DuckDB SQL list literal
    escaped = [p.replace("'", "''") for p in paths]
    return "[" + ",".join([f"'{p}'" for p in escaped]) + "]"

con = duckdb.connect(database=":memory:")
con.execute("PRAGMA threads=8")
con.execute("PRAGMA enable_object_cache")

print(con.execute("SELECT version()").fetchone()[0])


In [ ]:
# Create raw views over all CSVs in each family

con.execute(f"""
CREATE OR REPLACE VIEW raw_biometric AS
SELECT *, filename AS source_file
FROM read_csv_auto({_sql_str_list(FILES['biometric'])}, filename=true);
""")

con.execute(f"""
CREATE OR REPLACE VIEW raw_demographic AS
SELECT *, filename AS source_file
FROM read_csv_auto({_sql_str_list(FILES['demographic'])}, filename=true);
""")

con.execute(f"""
CREATE OR REPLACE VIEW raw_enrolment AS
SELECT *, filename AS source_file
FROM read_csv_auto({_sql_str_list(FILES['enrolment'])}, filename=true);
""")

# Row counts per family (includes enrolment duplicates if any)
counts = con.execute(
    """
    SELECT 'biometric' AS family, COUNT(*) AS n FROM raw_biometric
    UNION ALL SELECT 'demographic', COUNT(*) FROM raw_demographic
    UNION ALL SELECT 'enrolment', COUNT(*) FROM raw_enrolment
    """
).df()

counts


In [ ]:
# Normalize to a unified long table via UNPIVOT
# Note: the CSVs use dd-mm-yyyy, but read_csv_auto may parse to DATE; handle both.

con.execute(
    """
    CREATE OR REPLACE TABLE updates_long AS
    WITH
    bio AS (
        SELECT
            COALESCE(
                try_strptime(CAST(date AS VARCHAR), '%d-%m-%Y'),
                CAST(date AS TIMESTAMP)
            ) AS date,
            date_trunc('month', COALESCE(
                try_strptime(CAST(date AS VARCHAR), '%d-%m-%Y'),
                CAST(date AS TIMESTAMP)
            )) AS month,
            trim(state) AS state,
            trim(district) AS district_raw,
            lpad(CAST(pincode AS VARCHAR), 6, '0') AS pincode,
            'biometric' AS channel,
            replace(age_col, 'bio_age_', '') AS age_band,
            CAST(COALESCE(cnt, 0) AS BIGINT) AS count,
            source_file
        FROM raw_biometric
        UNPIVOT (cnt FOR age_col IN (bio_age_5_17, bio_age_17_))
    ),
    demo AS (
        SELECT
            COALESCE(
                try_strptime(CAST(date AS VARCHAR), '%d-%m-%Y'),
                CAST(date AS TIMESTAMP)
            ) AS date,
            date_trunc('month', COALESCE(
                try_strptime(CAST(date AS VARCHAR), '%d-%m-%Y'),
                CAST(date AS TIMESTAMP)
            )) AS month,
            trim(state) AS state,
            trim(district) AS district_raw,
            lpad(CAST(pincode AS VARCHAR), 6, '0') AS pincode,
            'demographic' AS channel,
            replace(age_col, 'demo_age_', '') AS age_band,
            CAST(COALESCE(cnt, 0) AS BIGINT) AS count,
            source_file
        FROM raw_demographic
        UNPIVOT (cnt FOR age_col IN (demo_age_5_17, demo_age_17_))
    ),
    enr AS (
        SELECT
            COALESCE(
                try_strptime(CAST(date AS VARCHAR), '%d-%m-%Y'),
                CAST(date AS TIMESTAMP)
            ) AS date,
            date_trunc('month', COALESCE(
                try_strptime(CAST(date AS VARCHAR), '%d-%m-%Y'),
                CAST(date AS TIMESTAMP)
            )) AS month,
            trim(state) AS state,
            trim(district) AS district_raw,
            lpad(CAST(pincode AS VARCHAR), 6, '0') AS pincode,
            'enrolment' AS channel,
            CASE
                WHEN age_col = 'age_0_5' THEN '0_5'
                WHEN age_col = 'age_5_17' THEN '5_17'
                WHEN age_col = 'age_18_greater' THEN '18_greater'
                ELSE replace(age_col, 'age_', '')
            END AS age_band,
            CAST(COALESCE(cnt, 0) AS BIGINT) AS count,
            source_file
        FROM raw_enrolment
        UNPIVOT (cnt FOR age_col IN (age_0_5, age_5_17, age_18_greater))
    )
    SELECT * FROM bio
    UNION ALL
    SELECT * FROM demo
    UNION ALL
    SELECT * FROM enr;
    """
)

# De-duplicate rows to prevent the extra enrolment folder from double-counting if it is a copy
con.execute(
    """
    CREATE OR REPLACE TABLE updates_long_dedup AS
    SELECT DISTINCT
        month, state, district_raw, pincode, channel, age_band, count
    FROM updates_long;
    """
)

con.execute(
    """
    SELECT
      (SELECT COUNT(*) FROM updates_long) AS rows_before,
      (SELECT COUNT(*) FROM updates_long_dedup) AS rows_after
    """
).df()


In [ ]:
# Add normalized keys for robust grouping (district_key collapses minor spacing/punctuation variants)
# Also apply basic data-quality filters (drop obviously malformed state/district/pincode).

con.execute("""
CREATE OR REPLACE TABLE updates_norm AS
SELECT
  month,
  trim(CAST(state AS VARCHAR)) AS state,
  trim(CAST(district_raw AS VARCHAR)) AS district_raw,
  regexp_replace(lower(trim(CAST(district_raw AS VARCHAR))), '[^a-z0-9]+', '', 'g') AS district_key,
  regexp_replace(lower(trim(CAST(state AS VARCHAR))), '[^a-z0-9]+', '', 'g') AS state_key,
  lpad(CAST(pincode AS VARCHAR), 6, '0') AS pincode,
  channel,
  age_band,
  count
FROM updates_long_dedup
WHERE
  state IS NOT NULL
  AND district_raw IS NOT NULL
  AND pincode IS NOT NULL
  AND length(trim(CAST(state AS VARCHAR))) > 1
  AND NOT regexp_matches(trim(CAST(state AS VARCHAR)), '^[0-9]+$')
  AND length(trim(CAST(district_raw AS VARCHAR))) > 1
  AND NOT regexp_matches(trim(CAST(district_raw AS VARCHAR)), '^[0-9]+$')
  AND regexp_matches(lpad(CAST(pincode AS VARCHAR), 6, '0'), '^[0-9]{6}$');
""")

con.execute("""
SELECT
  MIN(month) AS min_month,
  MAX(month) AS max_month,
  COUNT(DISTINCT state) AS n_states,
  COUNT(DISTINCT district_key) AS n_district_keys,
  COUNT(DISTINCT pincode) AS n_pincodes
FROM updates_norm
""").df()


In [ ]:
# Monthly cube with stream columns

con.execute("""
CREATE OR REPLACE TABLE cube_monthly AS
SELECT
  month,
  state,
  state_key,
  district_key,
  any_value(district_raw) AS district_raw,
  pincode,
  SUM(CASE WHEN channel='enrolment' AND age_band='0_5' THEN count ELSE 0 END) AS enrol_0_5,
  SUM(CASE WHEN channel='enrolment' AND age_band='5_17' THEN count ELSE 0 END) AS enrol_5_17,
  SUM(CASE WHEN channel='enrolment' AND age_band='18_greater' THEN count ELSE 0 END) AS enrol_18_plus,
  SUM(CASE WHEN channel='demographic' AND age_band='5_17' THEN count ELSE 0 END) AS demo_5_17,
  SUM(CASE WHEN channel='demographic' AND age_band='17_' THEN count ELSE 0 END) AS demo_17_plus,
  SUM(CASE WHEN channel='biometric' AND age_band='5_17' THEN count ELSE 0 END) AS bio_5_17,
  SUM(CASE WHEN channel='biometric' AND age_band='17_' THEN count ELSE 0 END) AS bio_17_plus
FROM updates_norm
GROUP BY month, state, state_key, district_key, pincode;
""")

cube = con.execute("""
SELECT * FROM cube_monthly
""").df()

cube['month'] = pd.to_datetime(cube['month'])

cube['total_updates'] = cube[[
  'enrol_0_5','enrol_5_17','enrol_18_plus','demo_5_17','demo_17_plus','bio_5_17','bio_17_plus'
]].sum(axis=1)

cube.shape, cube[['month','state','district_raw','pincode','total_updates']].head()


In [ ]:
STREAMS = ['enrol_0_5','enrol_5_17','enrol_18_plus','demo_5_17','demo_17_plus','bio_5_17','bio_17_plus']

def shannon_entropy_rows(x: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    # x: (n,k) nonnegative counts
    totals = x.sum(axis=1, keepdims=True)
    p = np.divide(x, totals, out=np.zeros_like(x, dtype=float), where=totals > 0)
    p = np.clip(p, eps, 1.0)
    h = -(p * np.log(p)).sum(axis=1)
    return h

X = cube[STREAMS].to_numpy(dtype=float)
k = X.shape[1]
cube['entropy'] = shannon_entropy_rows(X)
cube['entropy_norm'] = cube['entropy'] / np.log(k)

# Volume score (robust scaled within-state to reduce population scale differences)
cube['log_volume'] = np.log1p(cube['total_updates'])
cube['volume_rs'] = 0.0

for st, idx in cube.groupby('state').groups.items():
    rs = RobustScaler(with_centering=True, with_scaling=True, quantile_range=(25, 75))
    cube.loc[idx, 'volume_rs'] = rs.fit_transform(cube.loc[idx, ['log_volume']]).ravel()

# Temporal volatility: rolling std of log-volume + composition shift (L1 distance in proportions)
cube = cube.sort_values(['state_key','district_key','pincode','month']).reset_index(drop=True)

# proportions
P = np.divide(X, X.sum(axis=1, keepdims=True), out=np.zeros_like(X), where=X.sum(axis=1, keepdims=True) > 0)
for i, s in enumerate(STREAMS):
    cube[f'prop_{s}'] = P[:, i]

prop_cols = [f'prop_{s}' for s in STREAMS]

# Composition shift as total absolute change in composition vs previous month
cube['comp_shift'] = (
    cube.groupby(['state_key','district_key','pincode'])[prop_cols]
        .diff()
        .abs()
        .sum(axis=1)
        .fillna(0.0)
)

cube['vol_roll'] = (
    cube.groupby(['state_key','district_key','pincode'])['log_volume']
        .rolling(3, min_periods=2).std().reset_index(level=[0,1,2], drop=True)
).fillna(0.0)

# Robust normalize volatility within-state
cube['vol_rs'] = 0.0
for st, idx in cube.groupby('state').groups.items():
    rs = RobustScaler(with_centering=True, with_scaling=True, quantile_range=(25, 75))
    cube.loc[idx, 'vol_rs'] = rs.fit_transform(cube.loc[idx, ['vol_roll']]).ravel()

cube['comp_rs'] = 0.0
for st, idx in cube.groupby('state').groups.items():
    rs = RobustScaler(with_centering=True, with_scaling=True, quantile_range=(25, 75))
    cube.loc[idx, 'comp_rs'] = rs.fit_transform(cube.loc[idx, ['comp_shift']]).ravel()

# Composite IEI (tunable weights)
w_entropy, w_volume, w_vol, w_comp = 0.45, 0.20, 0.20, 0.15
cube['IEI'] = (
    w_entropy * cube['entropy_norm'] +
    w_volume  * (1 / (1 + np.exp(-cube['volume_rs']))) +
    w_vol     * (1 / (1 + np.exp(-cube['vol_rs']))) +
    w_comp    * (1 / (1 + np.exp(-cube['comp_rs'])))
)

cube[['month','state','district_raw','pincode','total_updates','entropy_norm','IEI']].head()


## Shock vs Lifecycle Interpretation (IEI delta logic)
Think of IEI as a **composite instability score**. The direction and shape of change matters more than the absolute level.

- **Lifecycle drift (slow)**: IEI increases gradually over multiple months, typically with modest `vol_spike` and small-to-moderate `comp_shift`. Interpretation: slow demographic/administrative churn (e.g., seasonal mobility, education flows).
- **Shock (fast)**: a **single-month jump** in IEI (large `d_IEI`) together with a strong `vol_spike` and often a sharp `comp_shift`. Interpretation: sudden disruption/campaign/displacement that changes both volume and mix quickly.
- **Stabilization**: IEI decreases (negative `d_IEI`) and volatility drops. Interpretation: post-episode normalization or administrative settling.

How to read the sign of `d_IEI`:
- `d_IEI > 0`: instability is increasing (more turbulent, more mixed, more volatile, or more composition shift).
- `d_IEI < 0`: instability is decreasing (signals becoming calmer / more routine).

Why we also look beyond `d_IEI`:
- IEI can rise because of **diversity/entropy** (mix) even if total updates are not huge.
- IEI can rise because of **volume** even if the mix is stable.
- The most informative episodes are usually those where **multiple components move together** (entropy + volume + volatility + composition shift).

## Why Entropy, Not Counts?
Raw counts tell you *how much* activity happened, but not *what kind* of activity happened. Entropy captures **diversity of update streams** (enrolment vs demographic vs biometric, and age bands), so it can detect administrative complexity and mix-shifts that simple totals miss. In practice: two regions can have similar total update volume but very different *composition*—and that difference often matters for interpreting whether the pattern looks like routine lifecycle churn, a documentation drive, or a disruptive shock.

In [ ]:
# One figure: entropy (mix) vs counts (volume) for the latest month

import pandas as pd
import plotly.express as px

latest_month = pd.to_datetime(cube['month']).max()
df_plot = cube.loc[cube['month'] == latest_month, ['state', 'district_raw', 'pincode', 'IEI', 'total_updates', 'entropy_norm']].copy()

# Keep plotting lightweight if there are many pincodes
n_max = 25_000
if len(df_plot) > n_max:
    df_plot = df_plot.sample(n=n_max, random_state=42)

df_plot['total_updates'] = pd.to_numeric(df_plot['total_updates'], errors='coerce')
df_plot['entropy_norm'] = pd.to_numeric(df_plot['entropy_norm'], errors='coerce')
df_plot = df_plot.dropna(subset=['total_updates', 'entropy_norm'])

fig = px.scatter(
    df_plot,
    x='total_updates',
    y='entropy_norm',
    hover_data=['state', 'district_raw', 'pincode', 'IEI'],
    opacity=0.35,
    title=f'Why entropy matters: same volume, different mix (latest month: {latest_month.date()})',
    labels={
        'total_updates': 'Total updates (volume)',
        'entropy_norm': 'Normalized entropy (mix diversity)',
    },
)
fig.update_xaxes(type='log')
fig.update_layout(height=560)
fig.show()


In [ ]:
# IEI summaries at State / District / Pincode

state_month = (
    cube.groupby(['state','month'], as_index=False)
        .agg(IEI=('IEI','mean'), total_updates=('total_updates','sum'))
)
district_month = (
    cube.groupby(['state','district_key','district_raw','month'], as_index=False)
        .agg(IEI=('IEI','mean'), total_updates=('total_updates','sum'))
)
pincode_month = cube[['state','district_key','district_raw','pincode','month','IEI','total_updates']].copy()

state_month.sort_values(['month','IEI'], ascending=[True, False]).head(10)


## District time series (same district)
Pick one district and view **total updates** and **IEI** over time (single combined plot).


In [ ]:
import re

import pandas as pd
import ipywidgets as widgets
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display

_STOPWORDS = {'and', 'of', 'the'}


def _singularize(token: str) -> str:
    if len(token) > 3 and token.endswith('s') and not token.endswith('ss'):
        return token[:-1]
    return token


def _key(s: object) -> str:
    """Normalize names for stable grouping (case/spacing/punctuation-insensitive)."""
    if s is None:
        return ''

    x = str(s).strip().lower().replace('&', ' and ')
    tokens = re.findall(r'[a-z0-9]+', x)
    tokens = [_singularize(t) for t in tokens if t not in _STOPWORDS]

    # Small abbreviation normalizations seen in practice
    if tokens[:1] == ['w'] and 'bengal' in tokens:
        tokens[0] = 'west'
    if tokens[:1] == ['wb']:
        tokens[0] = 'west'
        if 'bengal' not in tokens:
            tokens.append('bengal')

    return ''.join(tokens)


# Safety checks
required_cols = {'state', 'district_key', 'district_raw', 'month', 'IEI', 'total_updates'}
missing = required_cols - set(district_month.columns)
if missing:
    raise ValueError(f"district_month is missing columns: {sorted(missing)}")

# Work on a copy so we can add normalized keys
_dm = district_month.copy()
_dm['state'] = _dm['state'].astype(str).str.strip()
_dm['district_raw'] = _dm['district_raw'].astype(str).str.strip()
_dm['state_key'] = _dm['state'].map(_key)

# Build canonical state labels so the dropdown does not repeat casing variants
_state_labels = (
    _dm[_dm['state_key'] != '']
    .groupby('state_key')['state']
    .agg(lambda s: s.value_counts().index[0])
    .to_dict()
)

state_options = [(label, state_key) for state_key, label in sorted(_state_labels.items(), key=lambda kv: kv[1])]
if not state_options:
    raise ValueError('No states found in district_month.')

print(f"States in dropdown: {len(state_options)} (collapsed from {_dm['state'].nunique()} raw spellings)")

state_dd = widgets.Dropdown(
    options=state_options,
    value=state_options[0][1],
    description='State:',
    layout=widgets.Layout(width='420px'),
)

district_dd = widgets.Dropdown(options=[], description='District:', layout=widgets.Layout(width='520px'))


def _district_options_for_state_key(state_key_value: str) -> list[tuple[str, str]]:
    d = _dm.loc[_dm['state_key'] == state_key_value, ['district_key', 'district_raw']].drop_duplicates()
    d = d.sort_values(['district_raw', 'district_key'])

    # Option format: (label, value)
    return [(str(r['district_raw']), str(r['district_key'])) for _, r in d.iterrows()]


def _refresh_district_dropdown(*_):
    options = _district_options_for_state_key(state_dd.value)
    district_dd.options = options
    if options:
        district_dd.value = options[0][1]


def _plot_district_time_series(state_key_value: str, district_key_value: str) -> None:
    d = _dm[(_dm['state_key'] == state_key_value) & (_dm['district_key'] == district_key_value)].copy()
    if len(d) == 0:
        print('No data for the selected district.')
        return

    d['month'] = pd.to_datetime(d['month'])
    d = d.sort_values('month')

    state_label = _state_labels.get(state_key_value, state_key_value)
    district_name = d['district_raw'].mode().iloc[0] if 'district_raw' in d.columns else district_key_value

    fig = make_subplots(specs=[[{'secondary_y': True}]])

    fig.add_trace(
        go.Bar(
            x=d['month'],
            y=d['total_updates'],
            name='Total updates',
            marker_color='rgba(55, 126, 184, 0.55)',
        ),
        secondary_y=False,
    )

    fig.add_trace(
        go.Scatter(
            x=d['month'],
            y=d['IEI'],
            name='IEI (mean)',
            mode='lines+markers',
            line=dict(color='rgba(228, 26, 28, 0.95)', width=2),
        ),
        secondary_y=True,
    )

    fig.update_layout(
        title=f'District time series — {district_name} ({state_label})',
        height=520,
        barmode='overlay',
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0),
        margin=dict(l=40, r=40, t=70, b=40),
    )

    fig.update_xaxes(title_text='Month')
    fig.update_yaxes(title_text='Total updates', secondary_y=False)
    fig.update_yaxes(title_text='IEI (mean)', secondary_y=True, rangemode='tozero')

    fig.show()


_refresh_district_dropdown()
state_dd.observe(_refresh_district_dropdown, names='value')

ui = widgets.HBox([state_dd, district_dd])
display(ui)

out_plot = widgets.Output()
display(out_plot)


def _rerender(*_):
    with out_plot:
        out_plot.clear_output(wait=True)
        _plot_district_time_series(state_dd.value, district_dd.value)


district_dd.observe(_rerender, names='value')
state_dd.observe(_rerender, names='value')
_rerender()


In [ ]:
# Feature engineering for inference

feat = cube.copy()

def robust_z_series(x: pd.Series) -> pd.Series:
    med = x.median()
    mad = (x - med).abs().median()
    denom = 1.4826 * mad
    if not np.isfinite(denom) or denom < 1e-6:
        denom = 1.0
    return (x - med) / denom

# Deltas
for col in ['IEI','total_updates'] + [f'prop_{s}' for s in STREAMS]:
    feat[f'd_{col}'] = feat.groupby(['state_key','district_key','pincode'])[col].diff().fillna(0.0)

# Spike score for volume (robust z)
feat['vol_spike'] = (
    feat.groupby(['state_key','district_key','pincode'])['log_volume']
        .transform(robust_z_series)
        .fillna(0.0)
        .clip(-10, 10)
)

# Clip deltas to keep numerical stability (sign/direction is what matters for signatures)
for c in ['d_IEI','d_total_updates'] + [f'd_prop_{s}' for s in STREAMS]:
    if c in feat.columns:
        feat[c] = feat[c].clip(-5, 5)

FEATURES = [
    'entropy_norm','IEI','volume_rs','vol_rs','comp_rs','total_updates','log_volume','vol_spike',
    'enrol_0_5','enrol_5_17','enrol_18_plus','demo_5_17','demo_17_plus','bio_5_17','bio_17_plus'
] + [f'prop_{s}' for s in STREAMS] + [f'd_prop_{s}' for s in STREAMS] + ['d_IEI','d_total_updates']

# Fill NaNs safely
feat[FEATURES] = feat[FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0.0)

feat[['month','state','district_raw','pincode'] + FEATURES[:8]].head()


In [ ]:
# Event signatures: weight vectors over a compact subset of interpretable features.
# You can extend/adjust these; they're designed to be explainable.

SIGNATURE_FEATURES = [
    'd_total_updates','d_IEI','vol_spike','entropy_norm',
    'prop_enrol_0_5','prop_enrol_5_17','prop_enrol_18_plus',
    'prop_demo_5_17','prop_demo_17_plus',
    'prop_bio_5_17','prop_bio_17_plus',
    'd_prop_demo_17_plus','d_prop_bio_17_plus','d_prop_enrol_18_plus'
]

# Helper to build signatures
def sig(**weights):
    v = np.zeros(len(SIGNATURE_FEATURES), dtype=float)
    for k, w in weights.items():
        v[SIGNATURE_FEATURES.index(k)] = float(w)
    return v

EVENTS = {
    # Mobility / labor / urbanization
    'Migration_Surge': sig(d_total_updates=1.2, d_IEI=1.0, vol_spike=1.1, prop_demo_17_plus=0.8, d_prop_demo_17_plus=0.7, prop_enrol_18_plus=0.6),
    'Seasonal_Labor_Circulation': sig(d_total_updates=0.9, d_IEI=0.8, prop_demo_17_plus=0.6, d_prop_demo_17_plus=0.6, entropy_norm=0.4),
    'Urban_Spillover_PeriUrban': sig(d_IEI=0.9, prop_demo_17_plus=0.6, prop_enrol_18_plus=0.5, entropy_norm=0.7),
    'Job_Churn_Informal_Economy': sig(d_total_updates=1.0, d_IEI=1.0, entropy_norm=0.8, prop_demo_17_plus=0.7),
    # Household / marriage / restructuring
    'Marriage_Cluster_Household_Restructure': sig(d_total_updates=0.9, entropy_norm=0.9, prop_demo_17_plus=0.8, d_prop_demo_17_plus=0.6),
    'Household_Split_Merge': sig(entropy_norm=1.0, d_IEI=0.8, prop_demo_17_plus=0.6, prop_enrol_18_plus=0.4),
    # Education / children
    'Education_Pressure': sig(d_total_updates=0.9, prop_enrol_5_17=1.2, d_prop_enrol_18_plus=0.2, entropy_norm=0.6),
    'Child_Enrolment_Drive': sig(d_total_updates=1.0, prop_enrol_0_5=1.4, entropy_norm=0.5),
    'Student_Mobility': sig(d_total_updates=0.8, prop_enrol_5_17=1.0, prop_demo_17_plus=0.5, d_IEI=0.8),
    # Elderly / biometric friction
    'Elderly_Biometric_Refresh': sig(d_total_updates=0.7, prop_bio_17_plus=1.4, d_prop_bio_17_plus=0.9, entropy_norm=0.5),
    'Aging_Stress_Zone': sig(d_IEI=0.8, prop_bio_17_plus=1.2, prop_demo_17_plus=0.8, entropy_norm=0.6),
    # Administrative / campaigns
    'Admin_Cleanup_Campaign': sig(d_total_updates=1.1, entropy_norm=0.9, prop_demo_17_plus=0.6, prop_bio_17_plus=0.6),
    'ReKYC_Documentation_Drive': sig(d_total_updates=1.0, entropy_norm=0.8, prop_demo_17_plus=0.8),
    'System_Stabilization': sig(d_total_updates=-0.8, d_IEI=-1.0, vol_spike=-0.6, entropy_norm=-0.4),
    # Shocks / displacement (proxy)
    'Disaster_Displacement_Proxy': sig(d_total_updates=1.3, vol_spike=1.2, d_IEI=1.1, entropy_norm=0.9, prop_demo_17_plus=0.7),
    'Localized_Shock_Proxy': sig(vol_spike=1.4, d_IEI=1.2, d_total_updates=1.0),
    # Health/service accessibility proxies
    'Access_Friction_Biometric': sig(d_total_updates=0.6, prop_bio_17_plus=1.0, entropy_norm=0.6),
    # Baseline / mixed
    'Routine_Lifecycle': sig(entropy_norm=0.2, d_total_updates=0.1, d_IEI=0.1),
}

PRIORS = {
    # Mild priors; keep fairly flat to avoid overfitting
    k: 1.0 for k in EVENTS.keys()
}
PRIORS['Routine_Lifecycle'] = 1.5
PRIORS['System_Stabilization'] = 1.2

len(EVENTS), list(EVENTS)[:8]

## Event Ontology
The event library below is an **ontology** of population-level hypotheses. Each label corresponds to a distinct *pattern* in the feature space (volume, IEI delta, volatility spike, and shifts in age-band composition).

> Note: these are **probabilistic interpretations** of aggregated update signals, not ground-truth labels.

- **Migration_Surge** : broad adult-skewed update spike + rising IEI suggests sudden mobility/displacement pressure.
- **Seasonal_Labor_Circulation** : moderate recurrent churn with adult composition drift suggests cyclical labor movement.
- **Urban_Spillover_PeriUrban** : sustained adult-heavy mix + higher entropy suggests peri-urban assimilation churn.
- **Job_Churn_Informal_Economy** : higher entropy + rising IEI with adult tilt suggests unstable employment/addresses.
- **Marriage_Cluster_Household_Restructure** : higher entropy + adult share increase suggests household reconfiguration events.
- **Household_Split_Merge** : entropy led change with some IEI rise suggests administrative reshuffling across households.
- **Education_Pressure** : child/teen enrolment share rises alongside increased volume suggests schooling-driven updates.
- **Child_Enrolment_Drive** : strong 0–5 enrolment share rise suggests new-child enrolment campaigns or cohorts.
- **Student_Mobility** : teen share + IEI rise suggests student moves (hostels, admissions, relocations).
- **Elderly_Biometric_Refresh** : biometric 17+ share rise suggests biometric refresh/verification in older cohorts.
- **Aging_Stress_Zone** : adult/biometric-heavy composition with elevated IEI suggests aging-linked service friction.
- **Admin_Cleanup_Campaign** : broad multi-stream increase + higher entropy suggests programmatic cleanup/update drives.
- **ReKYC_Documentation_Drive** : documentation/demographic-heavy increase suggests re-KYC/documentation push.
- **System_Stabilization** : falling volume + falling IEI/volatility suggests normalization after a prior episode.
- **Disaster_Displacement_Proxy** : sharp multi-stream spike + high volatility suggests sudden disruption/displacement.
- **Localized_Shock_Proxy** : very high volatility spike (often short-lived) suggests localized administrative shock.
- **Access_Friction_Biometric** : biometric-heavy mix suggests service access friction concentrated in biometric updates.
- **Routine_Lifecycle** : weak, mixed signals; baseline lifecycle churn with low confidence by design.

In [ ]:
# Bayesian-style inference: posterior ∝ prior × exp(temperature × similarity)

def softmax(a: np.ndarray, axis: int = -1) -> np.ndarray:
    a = a - np.nanmax(a, axis=axis, keepdims=True)
    ea = np.exp(a)
    return ea / ea.sum(axis=axis, keepdims=True)

KEY_COLS = ['month','state','district_raw','district_key','pincode']

# Build matrix for signature features
sig_df = feat[KEY_COLS + SIGNATURE_FEATURES].copy()

M = sig_df[SIGNATURE_FEATURES].to_numpy(dtype=float)
M = np.nan_to_num(M, nan=0.0, posinf=0.0, neginf=0.0)
M = np.clip(M, -10, 10)

# Normalize evidence vectors (row-wise)
row_norm = np.linalg.norm(M, axis=1, keepdims=True)
M_unit = np.divide(M, row_norm, out=np.zeros_like(M), where=row_norm > 0)
M_unit = np.nan_to_num(M_unit, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

event_names = list(EVENTS.keys())
S = np.vstack([EVENTS[e] for e in event_names]).astype(float)
S = np.nan_to_num(S, nan=0.0, posinf=0.0, neginf=0.0)
S_norm = np.linalg.norm(S, axis=1, keepdims=True)
S_unit = np.divide(S, S_norm, out=np.zeros_like(S), where=S_norm > 0)
S_unit = np.nan_to_num(S_unit, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

# Similarity: cosine between evidence vector and signature vector
with np.errstate(divide='ignore', invalid='ignore', over='ignore'):
    sim = M_unit @ S_unit.T  # (n, n_events)

sim = np.nan_to_num(sim, nan=0.0, posinf=0.0, neginf=0.0)

temperature = 3.0
log_prior = np.log(np.array([PRIORS[e] for e in event_names], dtype=float) + 1e-12)
logits = temperature * sim + log_prior
post = softmax(logits, axis=1)

post_df = pd.DataFrame(post, columns=[f'P({e})' for e in event_names])

out = pd.concat([sig_df[KEY_COLS].reset_index(drop=True), post_df], axis=1)

# Confidence: max posterior + margin to second best
p_sorted = np.sort(post, axis=1)
out['p_max'] = p_sorted[:, -1]
out['p_margin'] = p_sorted[:, -1] - p_sorted[:, -2]
out['confidence'] = 0.65*out['p_max'] + 0.35*out['p_margin']

best_idx = post.argmax(axis=1)
out['event_top1'] = [event_names[i] for i in best_idx]

out.sort_values(['confidence'], ascending=False).head(20)


In [ ]:
# Hotspots: top inferred events per month at state level

state_events = (
    out.groupby(['month','state','event_top1'], as_index=False)
       .agg(confidence_mean=('confidence','mean'), rows=('p_max','size'))
       .sort_values(['month','confidence_mean'], ascending=[True, False])
)
state_events.head(15)

In [ ]:
# Interactive view: choose a month and see top events by state
month_choice = out['month'].max()
slice_df = state_events[state_events['month'] == month_choice].sort_values('confidence_mean', ascending=False)
fig = px.bar(slice_df, x='state', y='confidence_mean', color='event_top1',
             title=f'Top inferred life-event mix by state — {month_choice.date()}',
             hover_data=['rows'])
fig.update_layout(height=600)
fig.show()

## 6) Temporal clustering of inferred episodes
We cluster **high-confidence** region-month points (district+pincode) to identify episodes with similar signature dynamics.

This surfaces multi-region patterns that look like a coordinated shock or policy campaign.

In [ ]:
from hdbscan import HDBSCAN
import umap

# Select candidate points
cand_all = feat.merge(
    out[['month','state','district_key','pincode','event_top1','confidence']],
    on=['month','state','district_key','pincode'],
    how='inner'
)

# Adaptive confidence threshold (keeps clustering robust even if posteriors are diffuse)
threshold = 0.70
cand = cand_all[cand_all['confidence'] >= threshold].copy()
if len(cand) < 50:
    threshold = 0.55
    cand = cand_all[cand_all['confidence'] >= threshold].copy()

print(f"Candidates (confidence >= {threshold:.2f}):", len(cand))

CLUSTER_FEATURES = [
    'IEI','entropy_norm','vol_spike','comp_shift','log_volume',
    'prop_demo_17_plus','prop_bio_17_plus','prop_enrol_18_plus',
    'd_IEI','d_total_updates','d_prop_demo_17_plus','d_prop_bio_17_plus','d_prop_enrol_18_plus'
]

if len(cand) == 0:
    cand = cand_all.head(0).copy()
    cand['cluster'] = pd.Series(dtype=int)
    cand['umap_x'] = pd.Series(dtype=float)
    cand['umap_y'] = pd.Series(dtype=float)
else:
    cand[CLUSTER_FEATURES] = cand[CLUSTER_FEATURES].replace([np.inf, -np.inf], np.nan).fillna(0.0)
    Xc = RobustScaler().fit_transform(cand[CLUSTER_FEATURES].to_numpy(dtype=float))

    if len(cand) >= 50:
        embed = umap.UMAP(n_neighbors=25, min_dist=0.15, random_state=42).fit_transform(Xc)
        clusterer = HDBSCAN(min_cluster_size=25, min_samples=10)
        labels = clusterer.fit_predict(embed)
        cand['cluster'] = labels
        print('Clusters (excluding -1):', len(set(labels)) - (1 if -1 in labels else 0))
    else:
        cand['cluster'] = -1
        embed = np.zeros((len(cand), 2))

    cand['umap_x'] = embed[:, 0] if len(embed) else 0
    cand['umap_y'] = embed[:, 1] if len(embed) else 0

cand[['month','state','district_raw','pincode','event_top1','confidence','cluster']].head()


In [ ]:
# Cluster summary (what happened, where, when)

if len(cand) == 0 or 'cluster' not in cand.columns:
    cluster_summary = pd.DataFrame(
        columns=['cluster','n','month_min','month_max','states','top_event','conf_mean']
    )
else:
    cluster_summary = (
        cand[cand['cluster'] >= 0]
          .groupby('cluster', as_index=False)
          .agg(
              n=('cluster','size'),
              month_min=('month','min'),
              month_max=('month','max'),
              states=('state', lambda s: ', '.join(sorted(set(s))[:8]) + (' …' if len(set(s))>8 else '')),
              top_event=('event_top1', lambda s: s.value_counts().index[0]),
              conf_mean=('confidence','mean')
          )
          .sort_values(['conf_mean','n'], ascending=False)
    )

cluster_summary.head(20)


In [ ]:
# Visualize clusters in embedding space

if len(cand) == 0:
    print('No candidate points to cluster at current threshold.')
elif (cand['cluster'] >= 0).any():
    fig = px.scatter(
        cand,
        x='umap_x', y='umap_y',
        color=cand['cluster'].astype(str),
        hover_data=['month','state','district_raw','pincode','event_top1','confidence'],
        title='Episode clusters (UMAP + HDBSCAN)'
    )
    fig.update_layout(height=650)
    fig.show()
else:
    print('No stable clusters found yet; try lowering the threshold or reducing event library complexity.')


## 7) Outputs
We write clean, reusable artifacts to `outputs/`:
- `iei_pincode_month.parquet`
- `iei_state_month.parquet`
- `event_posteriors.parquet`
- `event_top_hotspots.csv` (easy to share)

In [ ]:
# Persist outputs

def robust_z(x: pd.Series) -> pd.Series:
    med = x.median()
    mad = (x - med).abs().median()
    denom = 1.4826 * mad
    if not np.isfinite(denom) or denom < 1e-6:
        denom = 1.0
    return (x - med) / denom

if 'IEI_rz' not in cube.columns:
    cube['IEI_rz'] = cube.groupby(['state_key','district_key','pincode'])['IEI'].transform(robust_z).fillna(0.0)

cube_out = cube[['month','state','district_raw','district_key','pincode','IEI','IEI_rz','entropy_norm','total_updates'] + STREAMS].copy()

cube_out.to_parquet(OUT_DIR / 'iei_pincode_month.parquet', index=False)
state_month.to_parquet(OUT_DIR / 'iei_state_month.parquet', index=False)
out.to_parquet(OUT_DIR / 'event_posteriors.parquet', index=False)
state_events.to_csv(OUT_DIR / 'event_top_hotspots.csv', index=False)
cluster_summary.to_csv(OUT_DIR / 'episode_clusters.csv', index=False)

stress_events = cube.loc[cube['IEI_rz'] >= 3.5, ['month','state','district_raw','pincode','IEI','IEI_rz','total_updates']].copy()
stress_events.sort_values(['IEI_rz'], ascending=False).to_csv(OUT_DIR / 'iei_stress_events.csv', index=False)

print('Wrote outputs to', OUT_DIR)
print('Stress events:', len(stress_events))


## 8) Executive summary (report view)
This section gives a quick, shareable overview similar to a report:
- Top **states / districts / pincodes** by IEI for the latest month
- **Stress** classification using robust z-score (`IEI_rz`) per (state, district, pincode) time series
- A few key plots (rankings, distribution, temporal trends)
- Top inferred **event mix by state** for the latest month

**Stress levels (based on `IEI_rz`)**
- `CRITICAL`: $IEI\_rz \ge 3.5$ (strong spike vs local history)
- `HIGH`: $2.5 \le IEI\_rz < 3.5$
- `ELEVATED`: $1.5 \le IEI\_rz < 2.5$
- `NORMAL`: $IEI\_rz < 1.5$

These are **screening thresholds**, not ground truth.

In [ ]:
# Report helpers + summary tables/plots

import numpy as np
import pandas as pd
import plotly.express as px


def robust_z(series: pd.Series) -> pd.Series:
    """Robust z-score using MAD; stable for heavy-tailed data."""
    median = series.median()
    mad = (series - median).abs().median()

    denom = 1.4826 * mad
    if not np.isfinite(denom) or denom < 1e-6:
        denom = 1.0

    return (series - median) / denom


def stress_label(z: float) -> str:
    if z >= 3.5:
        return 'CRITICAL'
    if z >= 2.5:
        return 'HIGH'
    if z >= 1.5:
        return 'ELEVATED'
    return 'NORMAL'


# Ensure we have IEI_rz available (per local region history)
if 'IEI_rz' not in cube.columns:
    cube['IEI_rz'] = (
        cube.groupby(['state_key', 'district_key', 'pincode'])['IEI']
        .transform(robust_z)
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
    )

latest_month = pd.to_datetime(cube['month']).max()
print('Latest month in data:', latest_month.date())

# --- Latest-month leaderboards ---
state_latest = (
    state_month[state_month['month'] == latest_month]
    .sort_values(['IEI', 'total_updates'], ascending=[False, False])
    .head(20)
    .reset_index(drop=True)
)

district_latest = (
    district_month[district_month['month'] == latest_month]
    .sort_values(['IEI', 'total_updates'], ascending=[False, False])
    .head(25)
    .reset_index(drop=True)
)

pincode_latest = (
    cube[cube['month'] == latest_month]
    .assign(stress=lambda d: d['IEI_rz'].astype(float).map(stress_label))
    .sort_values(['IEI_rz', 'IEI', 'total_updates'], ascending=[False, False, False])
    .loc[:, ['state', 'district_raw', 'pincode', 'IEI', 'IEI_rz', 'stress', 'total_updates']]
    .head(30)
    .reset_index(drop=True)
)

display(state_latest)
display(district_latest)
display(pincode_latest)

# --- Plots: rankings + distribution + volume relationship ---
if len(state_latest) > 0:
    fig = px.bar(
        state_latest.sort_values('IEI', ascending=True),
        x='IEI',
        y='state',
        orientation='h',
        title=f'Top states by IEI (latest month: {latest_month.date()})',
        hover_data=['total_updates'],
    )
    fig.update_layout(height=650)
    fig.show()

    fig = px.scatter(
        state_latest,
        x='total_updates',
        y='IEI',
        text='state',
        title=f'State volume vs IEI (latest month: {latest_month.date()})',
    )
    fig.update_traces(textposition='top center')
    fig.update_layout(height=520)
    fig.show()

# Distribution across states for latest month
state_dist = state_month.copy()
if len(state_dist) > 0:
    fig = px.histogram(
        state_dist[state_dist['month'] == latest_month],
        x='IEI',
        nbins=20,
        title=f'Distribution of state IEI (latest month: {latest_month.date()})',
    )
    fig.update_layout(height=450)
    fig.show()

# --- Temporal IEI trend (top states by average IEI) ---
top_states = (
    state_month.groupby('state', as_index=False)['IEI']
    .mean()
    .sort_values('IEI', ascending=False)
    .head(6)['state']
    .tolist()
)

trend = state_month[state_month['state'].isin(top_states)].sort_values(['state', 'month'])
if len(trend) > 0:
    fig = px.line(
        trend,
        x='month',
        y='IEI',
        color='state',
        markers=True,
        title='Temporal IEI trend (top 6 states by average IEI)',
    )
    fig.update_layout(height=520)
    fig.show()

# --- Latest-month inferred event mix by state (if available) ---
if 'state_events' in globals() and len(state_events) > 0:
    ev_latest = (
        state_events[state_events['month'] == latest_month]
        .sort_values(['confidence_mean', 'rows'], ascending=[False, False])
        .head(40)
    )

    if len(ev_latest) > 0:
        fig = px.bar(
            ev_latest,
            x='state',
            y='confidence_mean',
            color='event_top1',
            hover_data=['rows'],
            title=f'Inferred event mix by state (latest month: {latest_month.date()})',
        )
        fig.update_layout(height=650)
        fig.show()
else:
    print('state_events not found; run the inference cells above first.')


## Ethics and Privacy
- **Aggregated only**: all inputs are counts aggregated by month × geography × update stream; no individual records are used.
- **No identity inference**: this workflow does not attempt to identify, link, track, or re-identify any person.
- **Policy-safe**: outputs are framed as population-level *hypotheses* for planning and monitoring, not as enforcement signals.